# QEFM Plotting Notebook

## Imports

In [3]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import xarray as xr
from glob import glob
from pathlib import Path
import yaml
from tqdm import tqdm
import sys
import os

# Import refactored plotting functions
from load_stats import load_regional_stats_data, load_global_stats_data
from plots_regional import create_regional_plots
from plots_global import create_global_plots

## Constants
Levels: [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50] hPa

Variables:
- 3D variables: [H, Q, T, U, V]
- 2D variables: [P, PS]
- Slice variables: [Q2m, T2m, U10m, V10m, D2m]

Regions:
- NHE: Northern Hemisphere (20°N to 80°N)
- SHE: Southern Hemisphere (80°S to 20°S)
- TRO: Tropical Belt (20°S to 20°N)

In [4]:
# Configure start and end date of experiments
# Dates can be any date in the range of May 2024, or Dec 2024.
START_YEAR = "2024"
START_MONTH = "05"
START_DAY = "01"

END_YEAR = "2024"
END_MONTH = "05"
END_DAY = "01"

DATE_STR = f"{START_YEAR}{START_MONTH}{START_DAY}-{END_YEAR}{END_MONTH}{END_DAY}"

# We will look for stats in this directory; stats are stored in netcdf files
ROOT_DIR = "outputs"  # Where to look for stats
# Regional and global stats will be globbed
EXP0_BASE_PATTERN = "f5295fp_GEOSFP_MERRA2"  
# Regional and global stats will be globbed
EXP1_BASE_PATTERN = "ERA5_ERA5_ERA5" 
# Regional and global stats will be globbed
# EXP2_BASE_PATTERN = "Prithvi_ERA5_ERA5"
EXP2_BASE_PATTERN = "AIFS_ERA5_ERA5"

# Will only plot these variables across 3D and 2D collections
VARS_TO_PLOT = ["T"]

# Will only create level plots using these levels (zonal plots will still be created with all levels)
LEVELS_TO_PLOT = [850, 500, 200]

# Will only plot leads from this list; forecasts were made for up to 7 days
LEADS_TO_PLOT = [24, 72, 120]

# Percentiles to use for plotting color limits
PERCENTILES = [90, 65, 65]

# Choose from among [NHE, TRO, SHE]
# Regional stats will make plots for only these regions
REGIONS_TO_PLOT = ["NHE", "SHE", "TRO"]

## Get filenames of statistics netCDFs
These have already been created.

In [5]:
def get_stat_dataset_filenames(base_glob_patterns, root_dir, date_str):
    filename_dict = {"regional": [], "global": []}

    full_glob_patterns = [
        os.path.join(root_dir, f"stats*{base_pattern}_{date_str}*.nc4")
        for base_pattern in base_glob_patterns
    ]
    
    for pat in full_glob_patterns:
        stat_filenames = glob(pat)
        
        # Find regional and global files
        regional_files = [f for f in stat_filenames if "regional" in f]
        global_files = [f for f in stat_filenames if "global" in f]
        
        # Validate both exist
        if not regional_files or not global_files:
            missing = "regional" if not regional_files else "global"
            raise ValueError(f"No {missing} filename found for pattern: {pat}")
        
        filename_dict["regional"].append(regional_files)
        filename_dict["global"].append(global_files)
    
    return filename_dict

In [6]:
# Create list of all statistics filenames across all experiments
exp_patterns = [EXP0_BASE_PATTERN, EXP1_BASE_PATTERN, EXP2_BASE_PATTERN]
all_stat_filenames = get_stat_dataset_filenames(exp_patterns, ROOT_DIR, DATE_STR)
regional_stat_filenames = all_stat_filenames["regional"]
global_stat_filenames = all_stat_filenames["global"]

# There should be 1 file per experiment in each of regional, global filename lists
print(f"Regional stat files: {regional_stat_filenames}")
print(f"Global stat files: {global_stat_filenames}")

Regional stat files: [['outputs/stats_regional_f5295fp_GEOSFP_MERRA2_20240501-20240501_len10d_int6h_spc1d_91x144.nc4'], ['outputs/stats_regional_ERA5_ERA5_ERA5_20240501-20240501_len10d_int12h_spc1d_91x144.nc4'], ['outputs/stats_regional_AIFS_ERA5_ERA5_20240501-20240501_len10d_int12h_spc1d_91x144.nc4']]
Global stat files: [['outputs/stats_global_f5295fp_GEOSFP_MERRA2_20240501-20240501_len10d_int6h_spc1d_91x144.nc4'], ['outputs/stats_global_ERA5_ERA5_ERA5_20240501-20240501_len10d_int12h_spc1d_91x144.nc4'], ['outputs/stats_global_AIFS_ERA5_ERA5_20240501-20240501_len10d_int12h_spc1d_91x144.nc4']]


## Load important data from stats netcdf

### Regional

In [7]:
regional_data = load_regional_stats_data(
    regional_stat_filenames, 
    plot_RMS_decomp=False, 
    verbose=False, 
    vars_to_fetch=VARS_TO_PLOT
)

Starting loading of regional stat data from netcdf...
Found 3 experiment files
[['outputs/stats_regional_f5295fp_GEOSFP_MERRA2_20240501-20240501_len10d_int6h_spc1d_91x144.nc4'], ['outputs/stats_regional_ERA5_ERA5_ERA5_20240501-20240501_len10d_int12h_spc1d_91x144.nc4'], ['outputs/stats_regional_AIFS_ERA5_ERA5_20240501-20240501_len10d_int12h_spc1d_91x144.nc4']]
Loading single file for exp0...
Dataset contains 1 init dates from 20240501_00 to 20240501_00
Loading single file for exp1...
Dataset contains 1 init dates from 20240501_00 to 20240501_00
Loading single file for exp2...
Dataset contains 1 init dates from 20240501_00 to 20240501_00


ValueError: Experiment 2 has different regions than exp0.
  exp0: ['GLO', 'NHE', 'TRO', 'SHE', 'NPO', 'SPO', 'XPO', 'CUS', 'LND', 'NHL', 'TRL', 'SHL']
  exp2: ['GLO', 'NHE', 'TRO', 'SHE']

### Global

In [ ]:
# global_data_indiv = load_global_stats_data(
#     [stats_global_fn], YAML_FILENAME, process='comp', collections=None, load_raw=False, verbose=False
# )
global_data_comp = load_global_stats_data(
    global_stat_filenames, 
    process='comp', 
    load_raw=True, 
    verbose=True, 
    vars_to_fetch=VARS_TO_PLOT
)

# Stat Plotting

## Make output dir

In [ ]:
output_dir = Path(f"outputs/AIFS_2026_05_12")
output_dir.mkdir(exist_ok=True, parents=True)

## Regional stat plotting

In [ ]:
# Creates only per-level plots of ACC/RMSE
create_regional_plots(
    regional_data,
    vars_to_plot=VARS_TO_PLOT, 
    levels_to_plot=LEVELS_TO_PLOT,
    leads_to_plot=None,  # Lead filtering is nebulous
    regions_to_plot=REGIONS_TO_PLOT,
    stats_to_plot=[0, 1],  # ACC, RMSE only
    plots_dir=output_dir
)

## Global stat plotting

In [ ]:
create_global_plots(
    global_data_comp,
    vars_to_plot=VARS_TO_PLOT, 
    levels_to_plot=LEVELS_TO_PLOT,
    leads_to_plot=LEADS_TO_PLOT,
    exps_to_comp=[[2, 1], [2, 0]],
    limit_percentiles=PERCENTILES,
    colormap_key="ryb",
    gif_frame_duration=1000,
    output_dir=output_dir
)